In [1]:
!uv sync

Resolved 17 packages in 22ms
Audited 16 packages in 4ms


In [ ]:
# Imports
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from mandrop import (
    Params, PhysicalParams, SimulationParams,
    build, run, droplet_stats, plot_fields, compute_macros, boundary_stats,
)

print(f"JAX {jax.__version__}, devices: {jax.devices()}")

In [ ]:
# Configure the simulation
# Default PhysicalParams already match the real chip operating point:
#   300 µL/min oil, 40 µL/min center water, 20+20 µL/min sides
#   σ_eq = 5 mN/m (with surfactant), σ_clean = 20 mN/m (bare)
# Override here to study a different chemistry or operating point.
params = Params(
    physical = PhysicalParams(
        # u_oil_phys      = 0.32,    # m/s — flow rate per oil inlet
        # sigma_eq_phys   = 5e-3,    # N/m — equilibrium IFT
        # sigma_clean_phys= 20e-3,   # N/m — bare IFT
        # tau_ads_s       = 1e-4,    # s — surfactant adsorption timescale
    ),
    sim = SimulationParams(
        resolution_um = 2.5,         # 2.5 µm/lu for fast iteration; 1.0 for production
    ),
)
print(params.summary())

In [ ]:
# Build geometry + step function + initial state
geo, step, (f0, phi0, Gamma0), lat = build(params)
p = geo["params"]
Nx, Ny = p["Nx"], p["Ny"]
interior = geo["interior"]

print(f"Domain: {Nx}×{Ny}  Channel x∈[{p['gxL']},{p['gxR']}]  Throat x∈[{p['gxTL']},{p['gxTR']}]")
print(f"Probe point: ({p['probe_x']}, {p['probe_y']})  Outlet top y={p['Y_OUTLET_TOP']}")
print(f"Water prefill: {int(geo['water_prefill'].sum())} pixels")
print(f"Initial water (φ<0.5): {int((phi0 < 0.5).sum())} pixels")
print(f"Γ=1 (aged interface band): {int((Gamma0 > 0.5).sum())} pixels")

In [ ]:
# Inspect initial state: water prefill mask, relaxed φ, and Γ-aged band.
# The probe point (red dot in φ panel) is where droplet frequency is counted.
fig, axes = plt.subplots(1, 3, figsize=(12, 12))

axes[0].imshow(geo["water_prefill"].T, origin="lower", cmap="Blues", vmin=0, vmax=1)
axes[0].set_title("water_prefill (boolean)")

im1 = axes[1].imshow(phi0.T, origin="lower", cmap="RdBu", vmin=0, vmax=1)
axes[1].set_title("φ₀ (post relax_phi)")
axes[1].scatter([p["probe_x"]], [p["probe_y"]], c="red", s=60, marker="x", label="probe")
axes[1].axhline(p["Y_OUTLET_TOP"], color="k", ls=":", alpha=0.4, label="outlet top")
axes[1].axhline(p["Y_THROAT_BOT"], color="k", ls="--", alpha=0.4, label="throat bot")
axes[1].axhline(p["Y_THROAT_TOP"], color="k", ls="--", alpha=0.4, label="throat top")
axes[1].legend(loc="upper right", fontsize=7)
plt.colorbar(im1, ax=axes[1], shrink=0.5)

im2 = axes[2].imshow(Gamma0.T, origin="lower", cmap="magma", vmin=0, vmax=1)
axes[2].set_title(f"Γ₀ ({int((Gamma0 > 0.5).sum())} aged cells, rest = bare)")
plt.colorbar(im2, ax=axes[2], shrink=0.5)

for ax in axes: ax.set_aspect("equal")
plt.tight_layout(); plt.show()

# Sanity numbers
iface0 = (phi0 > 0.05) & (phi0 < 0.95)
print(f"prefill: {int(geo['water_prefill'].sum())} px  | water (φ<0.5): {int((phi0 < 0.5).sum())} px")
print(f"interface band (0.05<φ<0.95): {int(iface0.sum())} px  | Γ=1 on band: {int((Gamma0[iface0] > 0.5).sum())}")
print(f"probe at ({p['probe_x']}, {p['probe_y']})  initial φ_probe={float(phi0[p['probe_x'], p['probe_y']]):.3f}")

In [ ]:
# Visualize geometry and initial phi distribution
fig, axes = plt.subplots(1, 2, figsize=(8, 12))

im0 = axes[0].imshow(geo["wall"].T, origin='lower', cmap='gray_r', vmin=0, vmax=1)
axes[0].set_title('Wall mask')
axes[0].axhline(p['Y_USLOT_TOP'], color='b', ls=':', alpha=0.5, label='upper slots (water)')
axes[0].axhline(p['Y_USLOT_BOT'], color='b', ls=':', alpha=0.5)
axes[0].axhline(p['Y_LSLOT_TOP'], color='r', ls=':', alpha=0.5, label='lower slots (oil)')
axes[0].axhline(p['Y_LSLOT_BOT'], color='r', ls=':', alpha=0.5)
axes[0].legend(loc='upper right', fontsize=8)
plt.colorbar(im0, ax=axes[0], shrink=0.5)

im1 = axes[1].imshow(phi0.T, origin='lower', cmap='RdBu', vmin=0, vmax=1)
axes[1].set_title('Initial φ (oil=1, water=0)')
plt.colorbar(im1, ax=axes[1], shrink=0.5)

for ax in axes:
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# Run simulation, capture per-chunk droplet stats for trend tracking.
history = []
def collect(f, phi, Gamma, probe_history, step_num, wall_dt):
    s = droplet_stats(phi, p, probe_history=probe_history, dt_phys=lat.dt)
    history.append({"step": step_num, **s})
    print(f"  step {step_num:>6}  drops={s['n_drops']:>3}  d={s['d_mean_um']:>6.1f}µm  CV={s['d_cv']:>5.1%}  f={s['freq_Hz']:>6.0f} Hz")

f_final, phi_final, Gamma_final, total_steps = run(
    step, f0, phi0, Gamma0, interior, p,
    chunk_size=500, n_chunks=20,
    on_chunk=collect, verbose=False,
)

In [ ]:
# Summary statistics — final state + dimensional units via lat
rho_final, ux_final, uy_final = compute_macros(f_final)
vel_mag = jnp.sqrt(ux_final**2 + uy_final**2)
max_vel_phys = float(vel_mag.max()) * lat.dx / lat.dt
dp_pa = float(rho_final.max() - rho_final.min()) / 3.0 * lat.p_lu_to_pa
n_water = ((phi_final < 0.5).astype(jnp.float64) * interior).sum()
iface = (phi_final > 0.05) & (phi_final < 0.95)
gamma_mean = float(jnp.where(iface, Gamma_final, 0).sum() / jnp.maximum(iface.sum(), 1))

final_drops = droplet_stats(phi_final, p)

print(f"=== FLOW-FOCUSING — {total_steps} STEPS ===")
print(f"Wall-clock equivalent: {total_steps * lat.dt * 1000:.2f} ms physical")
print(f"Max velocity: {max_vel_phys:.3f} m/s  ({float(vel_mag.max()):.4e} lu/ts)")
print(f"Pressure drop: {dp_pa:.0f} Pa  (Δρ_lu={float(rho_final.max()-rho_final.min()):.4f})")
print(f"Water pixels:  {float(n_water):.0f}")
print(f"Γ_iface mean:  {gamma_mean:.3f}")
print()
print(f"=== DROPLETS ===")
print(f"Count in outlet: {final_drops['n_drops']}")
print(f"Mean diameter:   {final_drops['d_mean_um']:.1f} µm")
print(f"CV:              {final_drops['d_cv']:.1%}")
if len(final_drops['sizes_um']) > 0:
    print(f"Size spread:     {final_drops['sizes_um'].min():.1f} – {final_drops['sizes_um'].max():.1f} µm")

In [ ]:
# Final 5-panel: φ, ρ, u_y, |u|, Γ
plot_fields(f_final, phi_final, Gamma=Gamma_final)